# CSA08 Assignment 4 – Robust Data Processing with Files, Exceptions, Modules and Packages

**Student Name:** Darshan A  
**Register Number:** 1973260011  
**Course Code:** CSA0812  
**Task 1:** File-Processing Design Review: Diagnose and Repair a Poor Script  
**Marks:** 6 | **Bloom’s Level:** K4 – Analyse

---

## Aim
To analyse the weaknesses of a file-processing program and redesign it using safe file handling, line-by-line processing, specific exception handling and rejected-record tracking.

## Objectives
- Identify practical problems in the given mess-bill script.
- Develop a safer function that accepts a filename.
- Process records one line at a time.
- Handle invalid amounts and damaged records without stopping the complete process.
- Compare whole-file reading with line-by-line reading.


## Task 1.1 – Diagnose the Script (2 Marks)

| No. | Design deficiency | Possible problem in the hostel office |
|---|---|---|
| 1 | Hard-coded file path | The program may fail on another computer or when the file is moved. |
| 2 | File is not explicitly closed | The file resource may remain occupied longer than necessary. |
| 3 | Uses `read()` to load the complete file | A large file consumes more memory before processing begins. |
| 4 | Uses a bare `except` with `pass` | Different errors are hidden, making faults difficult to identify. |
| 5 | Converts the amount using `int()` | Amounts containing paise, such as 1250.50, cannot be processed. |
| 6 | Builds the report using repeated string concatenation | Repeated creation of strings is inefficient for a large report. |
| 7 | Does not handle blank lines safely | An empty line can produce missing fields and an indexing error. |
| 8 | Does not record rejected lines | Incorrect bills disappear from the process without an audit trail. |

**Diagnosis:** These weaknesses can cause incorrect totals, silent loss of records, poor auditability and unnecessary memory use.


## Create the Sample Input File

The following cell creates valid records, an invalid amount and a damaged record. It also writes actual newline characters between records so that each record is stored on a separate line in the text file.


In [1]:
# Create a sample mess bill input file

data = (
    "192311001,Rahul,1250.50\n"
    "192311002,Arun,1450.00\n"
    "192311003,Kiran,invalid\n"
    "192311004,Meena,980.75\n"
    "192311005,Divya\n"
    "192311006,Sam,1200.00\n"
)

with open("mess_oct.txt", "w", encoding="utf-8") as file:
    file.write(data)

print("Sample input file created successfully!")
print("\nInput file contents:")
print(data)


Sample input file created successfully!

Input file contents:
192311001,Rahul,1250.50
192311002,Arun,1450.00
192311003,Kiran,invalid
192311004,Meena,980.75
192311005,Divya
192311006,Sam,1200.00



## Task 1.2 – Repair the Design (3 Marks)

The repaired program uses a function with a filename parameter, a `with` statement, line-by-line reading, specific exceptions, `Decimal` for monetary accuracy, a report list and a rejected-lines list.


In [2]:
from decimal import Decimal, InvalidOperation


def process_mess_bills(filename):
    total = Decimal("0.00")
    report = []
    rejected_lines = []

    try:
        with open(filename, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):

                if not line.strip():
                    continue

                try:
                    parts = line.strip().split(",")

                    if len(parts) != 3:
                        raise ValueError("Expected exactly 3 fields")

                    register_number = parts[0].strip()
                    name = parts[1].strip()
                    amount = Decimal(parts[2].strip())

                    if not register_number or not name:
                        raise ValueError("Register number or name is missing")

                    if not amount.is_finite() or amount < 0:
                        raise ValueError("Invalid bill amount")

                    total += amount
                    report.append(
                        f"{register_number} {name} {amount:.2f}"
                    )

                except (ValueError, InvalidOperation) as error:
                    rejected_lines.append(
                        (line_number, line.rstrip(), str(error))
                    )

        print("MESS BILL REPORT")
        print("----------------")
        print("\n".join(report))
        print("\nTotal Mess Bill:", f"{total:.2f}")

        print("\nREJECTED RECORDS")
        print("----------------")
        for item in rejected_lines:
            print(item)

        return total, rejected_lines

    except FileNotFoundError:
        print("Error: Input file not found.")
    except PermissionError:
        print("Error: Permission denied.")

    return None, []


# Execute the corrected program
total, rejected_lines = process_mess_bills("mess_oct.txt")


MESS BILL REPORT
----------------
192311001 Rahul 1250.50
192311002 Arun 1450.00
192311004 Meena 980.75
192311006 Sam 1200.00

Total Mess Bill: 4881.25

REJECTED RECORDS
----------------
(3, '192311003,Kiran,invalid', "[<class 'decimal.ConversionSyntax'>]")
(5, '192311005,Divya', 'Expected exactly 3 fields')


## Verify the Result

The valid amounts are 1250.50, 1450.00, 980.75 and 1200.00.

**Expected total: 4881.25**

Two records should be rejected:
- Line 3: the amount is not a valid decimal number.
- Line 5: the record contains only two fields instead of three.


In [3]:
# Automatic result checks

assert total == Decimal("4881.25")
assert len(rejected_lines) == 2
assert rejected_lines[0][0] == 3
assert rejected_lines[1][0] == 5

print("All verification checks passed.")
print("Verified total:", f"{total:.2f}")
print("Number of rejected records:", len(rejected_lines))


All verification checks passed.
Verified total: 4881.25
Number of rejected records: 2


## Task 1.3 – Design Defence (1 Mark)

The corrected program is safer for processing 50,000 lines because it reads one line at a time instead of loading the entire file into memory. The `with` statement automatically closes the file after processing. Specific exception handling identifies invalid records without stopping the complete program. A list stores rejected records for future verification. Using `Decimal` also ensures accurate monetary calculations. Therefore, the program is more reliable and memory-efficient for large files.


## Alternative Concept and Complexity Analysis

If the entire file were read using `read()` or `readlines()`, all file contents would be loaded into memory before processing. The program would still need to examine each record, so the time complexity remains **O(n)** for both approaches, where `n` is the number of lines.

| Factor | Whole-file reading (`read()` / `readlines()`) | Line-by-line iteration |
|---|---|---|
| Time complexity | O(n) | O(n) |
| Additional reading memory | O(file size) | O(one line) |
| Processing approach | Loads all contents before processing | Processes each line as it is read |
| Suitability for large files | Uses more memory | More memory-efficient |

**Conclusion:** Line-by-line iteration is preferable for a file containing 50,000 lines because it avoids storing the complete input file in memory at once. The report and rejected-record lists still use memory for retained results, but the input-reading memory remains limited to approximately one line.


## Task 1 – Conclusion

The original script was improved by replacing the hard-coded path with a filename parameter, using a context manager, processing the file line by line, validating records, handling specific exceptions and retaining rejected records. The corrected program accurately totals valid mess bills and provides an audit trail for invalid records.
